In [2]:
from collections import defaultdict

def topological_sort(graph, n):
    visited = [False]*n
    stack = []

    def dfs(v):
        visited[v] = True
        for neighbor, _ in graph[v]: # Corrected line: added a space between '_' and 'in'
            if not visited[neighbor]:
                dfs(neighbor)
        stack.append(v)

    for i in range(n):
        if not visited[i]:
            dfs(i)

    return stack[::-1]


def shortest_path_dag(graph, n, src):
    INF = float('inf')
    dist = [INF]*n
    dist[src] = 0

    topo_order = topological_sort(graph, n)

    for u in topo_order:
        if dist[u] != INF:
            for v,w in graph[u]:
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w

    return dist


n = 6
graph = defaultdict(list)
graph[0].append((1,2))
graph[0].append((4, 1))
graph[1].append((2, 3))
graph[4].append((2, 2))
graph[4].append((5, 4))
graph[2].append((3, 6))
graph[5].append((3, 1))

dist = shortest_path_dag(graph, n, src = 0)
print("shortest paths desde nodo 0:")
for i, d in enumerate(dist):
    print(f"nodo {i}: {d}")

shortest paths desde nodo 0:
nodo 0: 0
nodo 1: 2
nodo 2: 3
nodo 3: 6
nodo 4: 1
nodo 5: 5


In [3]:
import bisect

def lis_dp(arr):
    n = len(arr)
    if n == 0:
        return 0, []

    dp = [1] * n
    parent = [-1] * n

    for i in range(1,n):
        for j in range(i):
            if arr[j] < arr[i] and dp[j] + 1 > dp[i]:
              dp[i] = dp[j] + 1
              parent[i] = j


    max_len = max(dp)
    idx = dp.index(max_len)

    subseq = []
    while idx != -1:
        subseq.append(arr[idx])
        idx = parent[idx]

    return max_len, subseq[::-1]


def lis_binary_search(arr):
    tails = []

    for x in arr:
        pos = bisect.bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)

        else:
            tails[pos] = x


    return len(tails)


arr = [10, 9, 2, 5, 3, 7, 101, 18]

length_dp, subseq = lis_dp(arr)
print(f"longitud = {length_dp}, subsecuencia = {subseq}")

length_bs = lis_binary_search(arr)
print(f"longitud = {length_bs}")


longitud = 4, subsecuencia = [2, 5, 7, 101]
longitud = 4


In [5]:
def knapsack_01(weights, values, capacity):
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i-1][w]
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i][w], dp[i-1][w - weights[i-1]] + values[i-1])

    selected = []
    w = capacity
    for i in range(n, 0 , -1):
        if dp[i][w] != dp[i-1][w]:
            selected.append(i-1)
            w -= weights[i-1]
    return dp[n][capacity], selected[::-1]



def knapsack_unbounded(weights, values, capacity):
    dp = [0] * (capacity + 1)

    for w in range(1, capacity + 1):
        for i in range(len(weights)):
            if weights[i] <= w:
                dp[w] = max(dp[w], dp[w - weights[i]] + values[i])
    return dp[capacity]


def knapsack_01_optimized(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for i in range(len(weights)):
        for w in range(capacity, weights[i] - 1, -1):
            dp[w] = max(dp[w], dp[w - weights[i]] + values[i])

    return dp[capacity]


# Example usage (add test data for execution)
weights = [10, 20, 30]
values = [60, 100, 120]
capacity = 50

val, items = knapsack_01(weights, values, capacity)
print("=== 0/1 knapssack ===")
print(f" valor maximo : {val}")
print(f"items usados: {items} pesos = {[weights[i] for i in items]}, valores = {[values[i] for i in items]}")

print("\n=== unbounded knapsack ===")
print(f"valor maximo : {knapsack_unbounded(weights, values, capacity)}")

print("\n=== 0/1 knapsack (espacio optimizado) ===")
print(f"valor maximo: {knapsack_01_optimized(weights, values, capacity)}")

=== 0/1 knapssack ===
 valor maximo : 220
items usados: [1, 2] pesos = [20, 30], valores = [100, 120]

=== unbounded knapsack ===
valor maximo : 300

=== 0/1 knapsack (espacio optimizado) ===
valor maximo: 220


In [8]:
import sys

def matrix_chain(dims):
    num_p = len(dims) # Number of dimension values (p0, p1, ..., pm)
    m = num_p - 1     # Number of matrices (A0, A1, ..., Am-1)

    dp = [[0] * m for _ in range(m)] # dp[i][j] for multiplying A_i to A_j
    split = [[0] * m for _ in range(m)]

    # Base case: length 1 (single matrix) has cost 0. dp[i][i] = 0 implicitly.

    for length in range(2, m + 1):  # length of chain of matrices (e.g., A_i, A_{i+1})
        for i in range(m - length + 1): # i is start index of matrix chain (0 to m-length)
            j = i + length - 1          # j is end index of matrix chain (i to m-1)
            dp[i][j] = sys.maxsize

            for k in range(i, j):       # k is split point: (A_i...A_k) * (A_{k+1}...A_j)
                # Cost calculation:
                # Dims of (A_i...A_k) are dims[i] x dims[k+1]
                # Dims of (A_{k+1}...A_j) are dims[k+1] x dims[j+1]
                # Cost of multiplying these two: dims[i] * dims[k+1] * dims[j+1]
                cost = dp[i][k] \
                     + dp[k + 1][j] \
                     + dims[i] * dims[k + 1] * dims[j + 1]

                if cost < dp[i][j]:
                    dp[i][j] = cost
                    split[i][j] = k

    # Nested build_order now uses m and 0-indexed matrices
    def build_order(i, j):
        if i == j:
            return f"A{i+1}" # A0 is A1, A1 is A2 etc.
        k = split[i][j]
        left = build_order(i, k)
        right = build_order(k + 1, j)
        return f"({left}{right})"

    # The result is for multiplying A_0 to A_{m-1}
    return dp[0][m-1], build_order(0, m - 1), dp, split


def print_dp_table(dims, dp_table, split_table):
    num_p = len(dims)
    m = num_p - 1 # Number of matrices

    print("Tabla DP costo minimo")

    header_cols = [f"A{x+1}" for x in range(m)] # A1 to Am
    print("   " + " ".join(f"{col:6}" for col in header_cols))

    for i in range(m): # i is 0-indexed matrix
        row = f"A{i+1} "
        for j in range(m): # j is 0-indexed matrix
            if j < i:
                row += "      - "
            else:
                row += f"{dp_table[i][j]:6} "
        print(row)

    print("\nTabla split (indice de corte k)")
    print("   " + " ".join(f"{col:6}" for col in header_cols))
    for i in range(m): # i is 0-indexed matrix
        row = f"A{i+1} "
        for j in range(m): # j is 0-indexed matrix
            if j < i:
                row += "      - "
            elif i == j:
                row += "      0 "
            else:
                row += f" k{split_table[i][j] + 1:4} " # k is also 0-indexed matrix, so +1 for A-label
        print(row)



dims = [30,35,15,5,10,20,25]

min_ops, orden, dp_result, split_result = matrix_chain(dims) # Capture all returned values

print("="*50)
print("CHAIN MATRIX MULTIPLICATION")
print("=" * 50)
print(f"\nMatrices:")
for i in range(len(dims) - 1):
    print(f"A{i + 1}: {dims[i]}x{dims[i+1]}")


print(f"\nCosto minimo: {min_ops:,} operaciones escalares")
print(f"Orden optimo: {orden}")

print()
print_dp_table(dims, dp_result, split_result)

CHAIN MATRIX MULTIPLICATION

Matrices:
A1: 30x35
A2: 35x15
A3: 15x5
A4: 5x10
A5: 10x20
A6: 20x25

Costo minimo: 15,125 operaciones escalares
Orden optimo: ((A1(A2A3))((A4A5)A6))

Tabla DP costo minimo
   A1     A2     A3     A4     A5     A6    
A1      0  15750   7875   9375  11875  15125 
A2       -      0   2625   4375   7125  10500 
A3       -       -      0    750   2500   5375 
A4       -       -       -      0   1000   3500 
A5       -       -       -       -      0   5000 
A6       -       -       -       -       -      0 

Tabla split (indice de corte k)
   A1     A2     A3     A4     A5     A6    
A1       0  k   1  k   1  k   3  k   3  k   3 
A2       -       0  k   2  k   3  k   3  k   3 
A3       -       -       0  k   3  k   3  k   3 
A4       -       -       -       0  k   4  k   5 
A5       -       -       -       -       0  k   5 
A6       -       -       -       -       -       0 
